In [1]:
import os
os.chdir('../')
%pwd


'c:\\Users\\YOGA\\OneDrive\\Documents\\AI Projects\\Text-Summarizer'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvalConfig:
    rootDir:Path
    dataPath:Path
    modelPath:Path
    tokenizerPath: Path
    metricFileName: Path
    

In [3]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import readYaml,createDir

In [4]:
class ConfigManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config=readYaml(config_path)
        

        createDir([self.config.artifactsRoots])

    def getModelEvalConfig(self)-> ModelEvalConfig:
        config= self.config.modelEvaluation
        
        
        createDir([config.rootDir])
        modelEvalConfig=ModelEvalConfig(
            rootDir=config.rootDir,
            dataPath=config.dataPath,
            modelPath=config.modelPath,
            tokenizerPath= config.tokenizerPath,
            metricFileName=config.metricFileName
                    
            
        )
        return modelEvalConfig


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import torch
from datasets import load_from_disk
import evaluate
from tqdm import tqdm
import pandas as pd

c:\Users\YOGA\OneDrive\Documents\AI Projects\Text-Summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class ModelEval:
    def __init__(self,config: ModelEvalConfig):
        self.config = config

    def generateBatchChunks(self, LoE, batchSize):
        for i in range(0, len(LoE), batchSize):
            yield LoE[i : i + batchSize]

    def calMetricTest(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batchSize=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        columnText="article",
        columnSummary="highlights",
    ):
        articleBatches = list(self.generateBatchChunks(dataset[columnText], batchSize))
        targetBatches = list(
            self.generateBatchChunks(dataset[columnSummary], batchSize)
        )

        for articleBatch, targetBatch in tqdm(
            zip(articleBatches, targetBatches), total=len(articleBatches)
        ):

            inputs = tokenizer(
                articleBatch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128,
            )
            """ parameter for length penalty ensures that the model does not generate sequences that are too long. """

            # Finally, we decode the generated texts,
            # replace the  token, and add the decoded texts with the references to the metric.
            decodedSummaries = [
                tokenizer.decode(
                    s, skip_special_tokens=True, clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]

            decodedSummaries = [d.replace("", " ") for d in decodedSummaries]

            metric.add_batch(predictions=decodedSummaries, references=targetBatch)

        #  Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer=AutoTokenizer.from_pretrained(self.config.tokenizerPath)
        modelPegasus= AutoModelForSeq2SeqLM.from_pretrained(self.config.modelPath).to(device)
        dfSampt= load_from_disk(self.config.dataPath)
        rougeNames = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rougeMetric = evaluate.load('rouge')

        score = self.calMetricTest(dfSampt['test'][0:10], rougeMetric, modelPegasus, tokenizer, batchSize = 2, columnText = 'dialogue', columnSummary= 'summary')

        # Directly use the scores without accessing fmeasure or mid
        rougeDict = {rn: score[rn] for rn in rougeNames}
        df = pd.DataFrame(rougeDict, index=[f'pegasus'])
        df.to_csv(self.config.metricFileName,index=False)

    

In [7]:
config = ConfigManager()
modelEvalConfig = config.getModelEvalConfig()
modelEvalutaion= ModelEval(config=modelEvalConfig)
modelEvalutaion.evaluate()

[2026-03-06 21:54:43,142: INFO: common: yaml file:config\config.yaml loaded sucessesful]
[2026-03-06 21:54:43,145: INFO: common: created dir at:artifacts]
[2026-03-06 21:54:43,150: INFO: common: created dir at:artifacts/modelEvaluation]


100%|██████████| 5/5 [08:24<00:00, 100.84s/it]


[2026-03-06 22:12:01,796: INFO: rouge_scorer: Using default tokenizer.]
